# Data Preprocessing: Cleaning Data for Machine Learning

Raw data is rarely ready for modeling. This notebook covers the essential techniques to transform messy real-world data into clean, ML-ready datasets.

## 1. Why Preprocessing Matters (Garbage In, Garbage Out)

Machine learning models make a variety of assumptions about input data:
- **Numerical stability**: Features with vastly different scales can cause gradient-based optimizers to oscillate.
- **Missing values**: Most sklearn estimators cannot handle NaN values directly.
- **Categorical data**: Models require numerical inputs.
- **Outliers**: Can skew learned parameters (especially in linear models).
- **Distribution assumptions**: Linear regression assumes normally distributed errors.

**Consequences of skipping preprocessing:**
- Model fails to converge or converges slowly.
- Biased or misleading feature importance.
- Poor generalization on unseen data.
- Runtime errors from incompatible data types.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import set_config

set_config(transform_output='pandas')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

print('Libraries loaded successfully.')

---
## 2. Handling Missing Values

Missing data appears as `NaN`, `None`, or placeholder values. The strategy depends on **missingness mechanisms**:
- **MCAR** (Missing Completely At Random): No systematic pattern — safe to drop or impute.
- **MAR** (Missing At Random): Systematic but explained by observed data — imputation works well.
- **MNAR** (Missing Not At Random): Systematic and not explained — needs careful handling.

In [ ]:
# Create a synthetic dataset with missing values
np.random.seed(42)
n = 500

df = pd.DataFrame({
    'age': np.random.randint(18, 70, n).astype(float),
    'income': np.random.lognormal(mean=10, sigma=0.6, size=n),
    'education_yrs': np.random.randint(8, 22, n).astype(float),
    'credit_score': np.random.normal(650, 50, n),
    'churn_risk': np.random.choice([0, 1], n, p=[0.7, 0.3]),
})

# Introduce missing values systematically
rng = np.random.default_rng(2024)
for col in ['age', 'income', 'education_yrs', 'credit_score']:
    mask = rng.random(n) < 0.12
    df.loc[mask, col] = np.nan

# Make income MNAR: high-income records more likely missing
income_high_mask = df['income'].notna() & (df['income'] > df['income'].quantile(0.8))
extra_miss = income_high_mask.sample(frac=0.25, random_state=42).index
df.loc[extra_miss, 'income'] = np.nan

print(f'Dataset shape: {df.shape}')
print(f'\nMissing counts:\n{df.isna().sum()}')
print(f'\nOverall missing rate: {(df.isna().sum().sum() / df.size):.1%}')

### 2.1 Visualizing Missing Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(df.isna(), cbar=False, cmap='viridis', ax=axes[0])
axes[0].set_title('Missing Data Heatmap (yellow = missing)')
axes[0].set_ylabel('Row index')

missing_pct = df.isna().mean().sort_values(ascending=False) * 100
bars = axes[1].barh(missing_pct.index, missing_pct.values, color='coral')
axes[1].bar_label(bars, fmt='%.1f%%')
axes[1].set_title('Missing Percentage per Feature')
axes[1].set_xlabel('% Missing')

plt.tight_layout()
plt.show()

### 2.2 Dropping Missing Values

In [ ]:
# Drop rows with any missing value — only when missing rate is very low (<1%)
df_dropped_rows = df.dropna()

# Drop columns with >40% missing
thresh = 0.4 * len(df)
cols_to_drop = df.columns[df.isna().sum() > thresh]
print(f'Rows after dropna(): {len(df_dropped_rows)} (was {len(df)})')
print(f'Row loss: {(1 - len(df_dropped_rows)/len(df)):.1%}')
print(f'Columns that would be dropped (>{40}% missing): {list(cols_to_drop)}')

### 2.3 Mean / Median / Mode Imputation

In [ ]:
from sklearn.impute import SimpleImputer

# Median imputer (robust to outliers)
median_imputer = SimpleImputer(strategy='median')
df_median_imputed = median_imputer.fit_transform(df.drop(columns='churn_risk'))

# Mode imputer for categorical (even though this is numeric)
mode_imputer = SimpleImputer(strategy='most_frequent')

print('Median imputed values:')
for col, val in zip(df.columns[:-1], median_imputer.statistics_):
    print(f'  {col}: {val:.2f}')

### 2.4 KNN Imputer (Advanced)

Uses k-nearest neighbors to impute missing values based on similarity across all features. Works well when features are correlated.

In [ ]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=5, weights='distance')
df_knn_imputed = knn_imputer.fit_transform(df.drop(columns='churn_risk'))

print('KNN Imputer — distance-weighted, 5 neighbors')
print(f'Shape: {df_knn_imputed.shape}')

### 2.5 Iterative Imputer (MICE)

Models each feature with missing values as a function of other features in a round-robin fashion. State-of-the-art for MAR data.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

mice_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42),
    max_iter=10,
    random_state=42
)
df_mice_imputed = mice_imputer.fit_transform(df.drop(columns='churn_risk'))

print('MICE Imputer — RandomForest estimator, 10 iterations')
print(f'Shape: {df_mice_imputed.shape}')

### 2.6 Indicator Columns for Missingness

Add binary indicators capturing *where* data was missing. Models can learn if missingness itself is predictive.

In [ ]:
from sklearn.impute import MissingIndicator

indicator = MissingIndicator(features='all')
indicator_features = indicator.fit_transform(df.drop(columns='churn_risk'))

indicator_df = pd.DataFrame(
    indicator_features,
    columns=[f'{col}_missing' for col in df.columns[:-1]],
    index=df.index
)

print(f'Indicator matrix shape: {indicator_df.shape}')
print(f'\nSample rows with any missing:')
indicator_df[indicator_df.any(axis=1)].head(5)

---
## 3. Feature Scaling

Many algorithms (SVM, KNN, PCA, neural nets) are sensitive to feature magnitude. Scaling ensures all features contribute equally.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, Normalizer

# Create a dataset with an outlier to demonstrate robustness
np.random.seed(42)
X_scaling = pd.DataFrame({
    'feature_a': np.random.exponential(scale=2, size=200),
    'feature_b': np.random.normal(50, 10, 200),
    'feature_c': np.random.uniform(0, 100, 200),
})
# Inject a severe outlier
X_scaling.iloc[0, 0] = 50
X_scaling.iloc[1, 1] = 200

scalers = {
    'Original': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler(),
    'MaxAbsScaler': MaxAbsScaler(),
    'Normalizer': Normalizer(),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (name, scaler) in zip(axes, scalers.items()):
    if scaler is None:
        data = X_scaling
    else:
        data = pd.DataFrame(scaler.fit_transform(X_scaling), columns=X_scaling.columns)
    for col in data.columns:
        ax.scatter(data.index, data[col], alpha=0.6, label=col, s=20)
    ax.set_title(name)
    ax.legend(fontsize=8)
    if name == 'Original':
        ax.set_ylim(-10, 210)

plt.suptitle('Comparing Scaling Methods on Data with Outliers', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### When to Use Which

| Scaler | Best For | Notes |
|---|---|---|
| **StandardScaler** | Normally distributed features | Zeros mean, unit variance |
| **MinMaxScaler** | Bounded features (e.g., pixels 0–255) | Sensitive to outliers |
| **RobustScaler** | Features with outliers | Uses median and IQR |
| **MaxAbsScaler** | Sparse data | Preserves sparsity (divides by max abs) |
| **Normalizer** | Text / row-wise magnitude matters | Scales rows to unit norm |

In [ ]:
# Summary statistics across scaling methods
summary = {}
for name, scaler in scalers.items():
    if scaler is None:
        data = X_scaling
    else:
        data = pd.DataFrame(scaler.fit_transform(X_scaling), columns=X_scaling.columns)
    summary[name] = {
        'Mean': data.mean().round(3).tolist(),
        'Std': data.std().round(3).tolist(),
        'Min': data.min().round(3).tolist(),
        'Max': data.max().round(3).tolist(),
    }

pd.DataFrame(summary).T

---
## 4. Encoding Categorical Variables

Models need numbers. Encoding maps categories to numeric representations.

In [ ]:
np.random.seed(42)
n_cat = 300

df_cat = pd.DataFrame({
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_cat,
                                  p=[0.3, 0.4, 0.2, 0.1]),
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix'], n_cat,
                              p=[0.35, 0.25, 0.2, 0.12, 0.08]),
    'target': np.random.normal(0, 1, n_cat),
})

print(df_cat.head(10))

### 4.1 Label Encoding & One-Hot Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

# Label Encoding (ordinal categories with inherent order)
label_enc = LabelEncoder()
education_encoded = label_enc.fit_transform(df_cat['education'])
print(f'Label encoding mapping: {dict(zip(label_enc.classes_, range(len(label_enc.classes_))))}')

# One-Hot Encoding (nominal — no order)
ohe = OneHotEncoder(sparse_output=False, drop='first')
city_ohe = ohe.fit_transform(df_cat[['city']])
print(f'One-hot columns: {ohe.get_feature_names_out().tolist()}')

### 4.2 Ordinal Encoding (Custom Order)

In [ ]:
ordinal_enc = OrdinalEncoder(
    categories=[['High School', 'Bachelor', 'Master', 'PhD']],
    dtype=int
)
education_ordinal = ordinal_enc.fit_transform(df_cat[['education']])
print(f'Ordinal encoding mapping: 0=High School, 1=Bachelor, 2=Master, 3=PhD')
df_cat['education_ordinal'] = education_ordinal
df_cat[['education', 'education_ordinal']].drop_duplicates().sort_values('education_ordinal')

### 4.3 Target / Mean Encoding

In [ ]:
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split

# Target encoding — replaces category with mean of target within that category
X_train, X_test, y_train, y_test = train_test_split(
    df_cat[['education', 'city']], df_cat['target'], test_size=0.2, random_state=42
)

target_enc = TargetEncoder(cols=['education', 'city'], smoothing=10.0)
X_train_te = target_enc.fit_transform(X_train, y_train)
X_test_te = target_enc.transform(X_test)

print('Target encoding — training set (first 5 rows):')
X_train_te.head()

### 4.4 Binary & Frequency Encoding

In [ ]:
from category_encoders import BinaryEncoder

# Binary Encoding: categories → binary digits → columns
binary_enc = BinaryEncoder(cols=['city'])
df_binary = binary_enc.fit_transform(df_cat[['city']])
print('Binary encoding of city:')
pd.concat([df_cat['city'].head(10), df_binary.head(10)], axis=1)

In [ ]:
# Frequency Encoding: replace category with its count/frequency
freq_map = df_cat['city'].value_counts(normalize=True)
df_cat['city_freq'] = df_cat['city'].map(freq_map)

print('Frequency encoding of city:')
df_cat[['city', 'city_freq']].drop_duplicates().sort_values('city_freq', ascending=False)

### Encoding Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1) Label encoding vs original
orig = df_cat['education'].value_counts()
encoded_counts = pd.Series(education_encoded).value_counts().sort_index()
axes[0].bar(orig.index, orig.values, color='steelblue', alpha=0.7)
axes[0].set_title('Original Categories (Education)')
axes[0].tick_params(axis='x', rotation=45)

# 2) One-hot encoded city (first 20 rows)
city_ohe_subset = pd.DataFrame(city_ohe[:20], columns=ohe.get_feature_names_out())
city_ohe_subset.plot(kind='bar', stacked=True, ax=axes[1], legend=False)
axes[1].set_title('One-Hot Encoding (first 20 rows)')
axes[1].set_xlabel('Row index')

# 3) Target encoding values
axes[2].barh(range(len(target_enc.mapping_)), [m['mean'].mean() for m in target_enc.mapping_.values()])
axes[2].set_yticks(range(len(target_enc.mapping_)))
axes[2].set_yticklabels(target_enc.mapping_.keys())
axes[2].set_title('Target Encoding Means')
axes[2].set_xlabel('Encoded value')

plt.suptitle('Encoding Methods Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Handling Outliers

Outliers can distort training. Detect them via Z-score or IQR, then decide: remove, cap, or transform.

In [ ]:
np.random.seed(42)
X_out = pd.DataFrame({
    'normal': np.random.normal(100, 15, 500),
    'skewed': np.random.exponential(scale=5, size=500),
})
# Inject outliers
X_out.iloc[::20, 0] = X_out.iloc[::20, 0] * 2.5
X_out.iloc[::15, 1] = X_out.iloc[::15, 1] * 4

### 5.1 Z-Score Method

In [ ]:
from scipy import stats

z_scores = np.abs(stats.zscore(X_out))
outlier_mask_z = (z_scores > 3).any(axis=1)

print(f'Z-score outliers (|z| > 3): {outlier_mask_z.sum()} ({outlier_mask_z.mean():.1%})')
print(f'\nOutlier rows:\n{X_out[outlier_mask_z].head()}')

### 5.2 IQR Method

In [ ]:
Q1 = X_out.quantile(0.25)
Q3 = X_out.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask_iqr = ((X_out < lower_bound) | (X_out > upper_bound)).any(axis=1)

print(f'IQR outliers (1.5×IQR beyond Q1/Q3): {outlier_mask_iqr.sum()} ({outlier_mask_iqr.mean():.1%})')
for col in X_out.columns:
    print(f'{col}: IQR = {IQR[col]:.2f}, bounds [{lower_bound[col]:.2f}, {upper_bound[col]:.2f}]')

### 5.3 Winsorization / Clipping

In [ ]:
from scipy.stats.mstats import winsorize

# Winsorize at 5th and 95th percentiles
X_winsorized = X_out.copy()
for col in X_winsorized.columns:
    low, high = np.percentile(X_winsorized[col], [5, 95])
    X_winsorized[col] = X_winsorized[col].clip(low, high)

# Compare original vs winsorized
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for idx, (col, ax) in enumerate(zip(X_out.columns, axes)):
    ax.boxplot([X_out[col], X_winsorized[col]], tick_labels=['Original', 'Winsorized'])
    ax.set_title(col)
    ax.set_ylabel('Value')

plt.suptitle('Winsorization Effect (clip at 5th / 95th percentile)', fontsize=13)
plt.tight_layout()
plt.show()

### 5.4 Log / Power Transformations for Outliers

In [ ]:
X_out['skewed_log'] = np.log1p(X_out['skewed'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(X_out['skewed'], bins=40, color='coral', edgecolor='white')
axes[0].set_title('Original (skewed)')
axes[1].hist(X_out['skewed_log'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Log Transformed')
for ax in axes:
    ax.set_ylabel('Frequency')
plt.suptitle('Log Transform Effect on Skewed Data with Outliers', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Feature Transformations

Make distributions more Gaussian, stabilize variance, or enforce bounds.

In [ ]:
np.random.seed(42)
X_trans = pd.DataFrame({
    'lognormal': np.random.lognormal(mean=1, sigma=0.8, size=1000),
    'exponential': np.random.exponential(scale=3, size=1000),
    'bimodal': np.concatenate([np.random.normal(-2, 0.5, 500), np.random.normal(3, 0.7, 500)]),
    'uniform': np.random.uniform(0, 10, 1000),
})

### 6.1 Log Transform

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(X_trans['lognormal'], bins=50, color='coral', edgecolor='white')
axes[0].set_title(f'Original (skewness={X_trans["lognormal"].skew():.2f})')

log_transformed = np.log(X_trans['lognormal'])
axes[1].hist(log_transformed, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title(f'Log Transform (skewness={log_transformed.skew():.2f})')

for ax in axes:
    ax.set_ylabel('Frequency')
plt.suptitle('Log Transform Effect on Skewed Data', fontsize=14)
plt.tight_layout()
plt.show()

### 6.2 Box-Cox Transform

Finds optimal λ to make data more Gaussian. Requires **positive** data.

In [ ]:
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox, probplot

bc = PowerTransformer(method='box-cox', standardize=False)
X_bc = bc.fit_transform(X_trans[['lognormal', 'exponential']])

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for idx, (col, ax_hist, ax_qq) in enumerate(zip(
    ['lognormal', 'exponential'],
    [axes[0, 0], axes[1, 0]],
    [axes[0, 1], axes[1, 1]]
)):
    orig = X_trans[col]
    trans = X_bc[:, idx]
    ax_hist.hist(trans, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax_hist.hist(orig, bins=50, color='coral', edgecolor='white', alpha=0.4)
    ax_hist.set_title(f'{col} — Box-Cox (λ={bc.lambdas_[idx]:.3f})')
    
    stats.probplot(trans, dist='norm', plot=ax_qq)
    ax_qq.get_lines()[0].set_markersize(2)
    ax_qq.get_lines()[1].set_color('red')
    ax_qq.set_title(f'{col} — Q-Q Plot')

plt.suptitle('Box-Cox Transform Demo', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 6.3 Yeo-Johnson Transform

Like Box-Cox but **handles zero and negative values**.

In [ ]:
yj = PowerTransformer(method='yeo-johnson', standardize=False)
X_yj = yj.fit_transform(X_trans[['bimodal', 'uniform']])

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for idx, (col, ax_orig, ax_trans) in enumerate(zip(
    ['bimodal', 'uniform'],
    [axes[0, 0], axes[0, 1]],
    [axes[1, 0], axes[1, 1]]
)):
    ax_orig.hist(X_trans[col], bins=50, color='coral', edgecolor='white')
    ax_orig.set_title(f'{col} — Original')
    ax_trans.hist(X_yj[:, idx], bins=50, color='steelblue', edgecolor='white')
    ax_trans.set_title(f'{col} — Yeo-Johnson (λ={yj.lambdas_[idx]:.3f})')

plt.suptitle('Yeo-Johnson Transform (works with negative values)', fontsize=14)
plt.tight_layout()
plt.show()

### 6.4 Quantile Transform

Maps data to a uniform or normal distribution using percentiles. Non-linear, but preserves rank order.

In [ ]:
from sklearn.preprocessing import QuantileTransformer

qt_uniform = QuantileTransformer(output_distribution='uniform', n_quantiles=100)
qt_normal = QuantileTransformer(output_distribution='normal', n_quantiles=100)

X_qt_uniform = qt_uniform.fit_transform(X_trans[['exponential']])
X_qt_normal = qt_normal.fit_transform(X_trans[['exponential']])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(X_trans['exponential'], bins=50, color='coral', edgecolor='white')
axes[0].set_title('Original')
axes[1].hist(X_qt_uniform, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Quantile → Uniform')
axes[2].hist(X_qt_normal, bins=50, color='steelblue', edgecolor='white')
axes[2].set_title('Quantile → Normal')
for ax in axes:
    ax.set_ylabel('Frequency')
plt.suptitle('Quantile Transform on Exponential Data', fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. Discretization / Binning

Convert continuous features into categorical bins. Useful for interpretability and non-linear patterns.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

np.random.seed(42)
ages = np.random.randint(18, 80, 200).reshape(-1, 1)

strategies = ['uniform', 'quantile', 'kmeans']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, strat in zip(axes, strategies):
    kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy=strat)
    binned = kbd.fit_transform(ages).flatten()
    ax.scatter(ages.flatten(), binned, alpha=0.6, s=30)
    ax.set_title(f'KBins strategy: {strat}')
    ax.set_xlabel('Age')
    ax.set_ylabel('Bin ID')
    ax.set_yticks(range(5))

plt.suptitle('Discretization Strategies Comparison', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Creating Polynomial Features (Interaction Terms)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

X_poly = pd.DataFrame({
    'feature_1': np.random.randn(100),
    'feature_2': np.random.randn(100),
})

poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly_transformed = poly.fit_transform(X_poly)

print('Original features:', X_poly.columns.tolist())
print('Polynomial features (degree=2):', poly.get_feature_names_out().tolist())
print(f'\nNew shape: {X_poly_transformed.shape} (was {X_poly.shape})')
X_poly_transformed.head(8)

---
## 9. Text Preprocessing Basics

Raw text → cleaned tokens → numerical features.

In [ ]:
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

corpus = [
    'ML models love clean data!',
    'Garbage in == garbage out.',
    'Feature engineering is 80% of the work.',
    'Preprocessing pipelines save time and bugs.',
    'Missing values? Impute them wisely.',
]

# Simple text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.strip()

cleaned_corpus = [clean_text(doc) for doc in corpus]
print('Original vs cleaned:')
for orig, clean in zip(corpus, cleaned_corpus):
    print(f'  {orig:<40} → {clean}')

# Bag-of-Words
vectorizer = CountVectorizer(stop_words='english')
bow = vectorizer.fit_transform(cleaned_corpus)
print(f'\nBoW shape: {bow.shape}')
print(f'Vocabulary: {list(vectorizer.vocabulary_.keys())}')

---
## 10. Date / Time Feature Extraction

In [ ]:
np.random.seed(42)
date_rng = pd.date_range(start='2020-01-01', end='2024-12-31', periods=1000)
df_dates = pd.DataFrame({'timestamp': np.random.choice(date_rng, 500)})

df_dates['year'] = df_dates['timestamp'].dt.year
df_dates['month'] = df_dates['timestamp'].dt.month
df_dates['day'] = df_dates['timestamp'].dt.day
df_dates['dayofweek'] = df_dates['timestamp'].dt.dayofweek
df_dates['quarter'] = df_dates['timestamp'].dt.quarter
df_dates['is_weekend'] = df_dates['dayofweek'].isin([5, 6]).astype(int)
df_dates['dayofyear'] = df_dates['timestamp'].dt.dayofyear
df_dates['elapsed_days'] = (df_dates['timestamp'] - pd.Timestamp('2020-01-01')).dt.days

print('Engineered date features:')
df_dates.head(10)

---
## 11. Putting It All Together: Full Preprocessing Pipeline

Use `ColumnTransformer` to apply different transformations to different columns.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

np.random.seed(42)
n_samples = 1000

df_full = pd.DataFrame({
    'age': np.random.randint(18, 70, n_samples).astype(float),
    'income': np.random.lognormal(mean=10, sigma=0.5, size=n_samples),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples,
                                   p=[0.3, 0.4, 0.2, 0.1]),
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix'], n_samples,
                              p=[0.35, 0.25, 0.2, 0.12, 0.08]),
    'signup_date': np.random.choice(pd.date_range('2020-01-01', '2024-12-31'), n_samples),
    'churn': np.random.choice([0, 1], n_samples, p=[0.7, 0.3]),
})

# Inject missing values
for col in ['age', 'income']:
    mask = np.random.random(n_samples) < 0.1
    df_full.loc[mask, col] = np.nan

df_full.head(5)

In [ ]:
# Define preprocessing pipelines per column type
numeric_features = ['age', 'income']
categorical_features = ['education', 'city']
datetime_features = ['signup_date']

numeric_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=3)),
    ('scaler', RobustScaler()),
])

categorical_pipeline = Pipeline([
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')),
])

def extract_datetime_features(X):
    X = X.copy()
    ts = pd.to_datetime(X.iloc[:, 0])
    return pd.DataFrame({
        'year': ts.dt.year,
        'month': ts.dt.month,
        'day': ts.dt.day,
        'dayofweek': ts.dt.dayofweek,
        'elapsed_days': (ts - pd.Timestamp('2020-01-01')).dt.days,
    })

from sklearn.preprocessing import FunctionTransformer
datetime_pipeline = Pipeline([
    ('extract', FunctionTransformer(extract_datetime_features)),
    ('scale', StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
        ('datetime', datetime_pipeline, datetime_features),
    ],
    verbose_feature_names_out=True,
)

# Transform
X = df_full.drop('churn', axis=1)
y = df_full['churn']

X_transformed = preprocessor.fit_transform(X)
print(f'Transformed shape: {X_transformed.shape}')
print(f'Feature names:\n  {X_transformed.columns.tolist()}')
X_transformed.head(10)

In [ ]:
# Full model pipeline
full_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42)),
])

# Cross-validation
scores = cross_val_score(full_pipeline, X, y, cv=5, scoring='roc_auc')
print(f'Cross-validated ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}')

### Before / After Preprocessing Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Before: numeric features
for idx, feat in enumerate(numeric_features):
    axes[0, idx].hist(X[feat].dropna(), bins=40, color='coral', edgecolor='white', alpha=0.7)
    axes[0, idx].set_title(f'Before: {feat}')

# After: numeric features
for idx, feat in enumerate(numeric_features):
    axes[1, idx].hist(X_transformed[[f'pipeline__numeric__{feat}']].dropna(), bins=40,
                      color='steelblue', edgecolor='white', alpha=0.7)
    axes[1, idx].set_title(f'After: {feat} (imputed + RobustScaled)')

# Categorical: before vs after cardinality change
cat_original = X['education'].value_counts()
axes[0, 2].barh(cat_original.index, cat_original.values, color='coral', alpha=0.7)
axes[0, 2].set_title('Before: education (categories)')

encoded_cols = [c for c in X_transformed.columns if 'education' in c]
encoded_means = X_transformed[encoded_cols].mean()
axes[1, 2].barh(encoded_means.index, encoded_means.values, color='steelblue', alpha=0.7)
axes[1, 2].set_title('After: education (one-hot encoded)')

plt.suptitle('Before vs After Preprocessing Pipeline', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 12. Practice Exercises

Try these to reinforce what you've learned:

**Exercise 1: Missing Data Strategy**

Load the `titanic` dataset from seaborn. Identify columns with >5% missing values. For each, decide whether to drop, impute (and with what strategy), or add an indicator column. Build a `ColumnTransformer` that implements your strategy and apply it.

**Exercise 2: Scaling Showdown**

Generate a dataset with 4 features: one normally distributed, one exponential, one uniform, and one with 5% extreme outliers. Apply `StandardScaler`, `MinMaxScaler`, `RobustScaler`, and `MaxAbsScaler`. Fit a KNN classifier on each scaled version and compare ROC-AUC on a test set. Which scaler performs best and why?

**Exercise 3: Encoding Comparison**

Take the `diamonds` dataset from seaborn. Encode the `cut` and `color` columns using: One-Hot, Ordinal, Target, Binary, and Frequency encoding. Train a Ridge regression to predict `price` for each encoding. Compare R² scores and discuss why a particular encoding works best for `cut` vs `color`.

**Exercise 4: Outlier Treatment Thresholds**

Create a synthetic dataset with a single feature having 3% injected outliers. Compare the following approaches by measuring RMSE on a clean holdout set: (a) no treatment, (b) drop Z-score outliers at thresholds 2, 3, 4, (c) winsorization at percentiles 1/99, 5/95, 10/90, (d) log transform, (e) Box-Cox transform. Visualize results.

**Exercise 5: Full Pipeline Challenge**

Build an end-to-end preprocessing pipeline for the `mpg` dataset from seaborn. Requirements:
- Impute missing `horsepower` with KNN Imputer.
- Scale `weight` and `displacement` with robust scaling.
- Extract `origin` as a feature (it's encoded as 1/2/3 — treat it as categorical with proper encoding).
- Bin `mpg` into 4 quantile bins as the target variable.
- Train a GradientBoostingClassifier and report accuracy on a test set. Visualize the test confusion matrix.

---
*Notebook complete. Remember: good preprocessing is often more impactful than model choice.*